In [19]:
import pandas as pd

train_df = pd.read_csv('../data/processed_data/train.csv')
test_df = pd.read_csv('../data/processed_data/test.csv')

train_df.head()

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,port_long,lat_diff_to_port_1step,long_diff_to_port_1step,abs_lat_diff_to_port_1step,abs_long_diff_to_port_1step,lat_approach_rate,long_approach_rate,moving_towards_port_lat,moving_towards_port_long,euclidean_dist_to_port
0,0.031663,0.858217,17.1,-6,316,0,01-08 06:00,7.50361,77.58340,61e9f38eb937134a3c4bfd8b,...,80.341111,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
1,0.031707,0.856825,17.3,5,313,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,...,72.885278,-11.438334,4.698122,11.438334,4.698122,NaN,NaN,0,0,12.365591
2,0.031757,0.854596,16.9,5,312,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,...,72.885278,-11.368924,4.609772,11.368924,4.609772,-0.006105,-0.019166,1,1,12.267943
3,0.031798,0.857660,16.9,6,313,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,...,72.885278,-11.291514,4.508762,11.291514,4.508762,-0.006856,-0.022403,1,1,12.158422
4,0.031838,0.855153,16.3,7,313,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,...,72.885278,-11.229194,4.428662,11.229194,4.428662,-0.005550,-0.018087,1,1,12.070950


In [20]:
print(train_df.columns)

Index(['time', 'cog', 'sog', 'rot', 'heading', 'navstat', 'etaRaw', 'latitude',
       'longitude', 'vesselId', 'portId', 'latitude_1_steps_ago',
       'longitude_1_steps_ago', 'time_position_1_steps_ago',
       'latitude_2_steps_ago', 'longitude_2_steps_ago',
       'time_position_2_steps_ago', 'latitude_3_steps_ago',
       'longitude_3_steps_ago', 'time_position_3_steps_ago',
       'latitude_4_steps_ago', 'longitude_4_steps_ago',
       'time_position_4_steps_ago', 'latitude_5_steps_ago',
       'longitude_5_steps_ago', 'time_position_5_steps_ago',
       'max_lat_change_last_5_steps', 'min_lat_change_last_5_steps',
       'max_long_change_last_5_steps', 'min_long_change_last_5_steps',
       'cog_1_step_ago', 'time_cog_1_step_ago', 'cog_2_steps_ago',
       'time_cog_2_steps_ago', 'hours_passed', 'hour_sin', 'hour_cos',
       'minute_sin', 'minute_cos', 'month_of_the_year', 'week_of_the_year',
       'day_of_the_year', 'day_of_the_month', 'day_of_the_week',
       'hour_of_the_

In [21]:
import pandas as pd
import numpy as np

# Sort by vesselId and time to calculate distances between consecutive positions
train_df = train_df.sort_values(by=['vesselId', 'time']).reset_index(drop=True)

# Step 2: Calculate the changes between positions
train_df['lat_change_1_steps'] = train_df['latitude'] - train_df['latitude_1_steps_ago']
train_df['lat_change_2_steps'] = train_df['latitude'] - train_df['latitude_2_steps_ago']
train_df['lat_change_3_steps'] = train_df['latitude'] - train_df['latitude_3_steps_ago']
train_df['lat_change_4_steps'] = train_df['latitude'] - train_df['latitude_4_steps_ago']
train_df['lat_change_5_steps'] = train_df['latitude'] - train_df['latitude_5_steps_ago']


train_df['lon_change_1_steps'] = train_df['longitude'] - train_df['longitude_1_steps_ago']
train_df['lon_change_2_steps'] = train_df['longitude'] - train_df['longitude_2_steps_ago']
train_df['lon_change_3_steps'] = train_df['longitude'] - train_df['longitude_3_steps_ago']
train_df['lon_change_4_steps'] = train_df['longitude'] - train_df['longitude_4_steps_ago']
train_df['lon_change_5_steps'] = train_df['longitude'] - train_df['longitude_5_steps_ago']


train_df['lat_change_2_to_1_steps'] = train_df['latitude_1_steps_ago'] - train_df['latitude_2_steps_ago']
train_df['lat_change_3_to_2_steps'] = train_df['latitude_2_steps_ago'] - train_df['latitude_3_steps_ago']
train_df['lat_change_4_to_3_steps'] = train_df['latitude_3_steps_ago'] - train_df['latitude_4_steps_ago']
train_df['lat_change_5_to_4_steps'] = train_df['latitude_4_steps_ago'] - train_df['latitude_4_steps_ago']

train_df['lon_change_2_to_1_steps'] = train_df['longitude_1_steps_ago'] - train_df['longitude_2_steps_ago']
train_df['lon_change_3_to_2_steps'] = train_df['longitude_2_steps_ago'] - train_df['longitude_3_steps_ago']
train_df['lon_change_4_to_3_steps'] = train_df['longitude_3_steps_ago'] - train_df['longitude_4_steps_ago']
train_df['lon_change_5_to_4_steps'] = train_df['longitude_4_steps_ago'] - train_df['longitude_5_steps_ago']



# Step 3: Calculate the average latitude change for each vesselId based on 1-step and 2-step changes
avg_lat_change_1_steps = train_df.groupby('vesselId')['lat_change_1_steps'].mean().reset_index()
avg_lat_change_1_steps.columns = ['vesselId', 'avg_lat_change_1_steps']

avg_lat_change_2_steps = train_df.groupby('vesselId')['lat_change_2_steps'].mean().reset_index()
avg_lat_change_2_steps.columns = ['vesselId', 'avg_lat_change_2_steps']

avg_lat_change_3_steps = train_df.groupby('vesselId')['lat_change_3_steps'].mean().reset_index()
avg_lat_change_3_steps.columns = ['vesselId', 'avg_lat_change_3_steps']

avg_lat_change_4_steps = train_df.groupby('vesselId')['lat_change_4_steps'].mean().reset_index()
avg_lat_change_4_steps.columns = ['vesselId', 'avg_lat_change_4_steps']

avg_lat_change_5_steps = train_df.groupby('vesselId')['lat_change_5_steps'].mean().reset_index()
avg_lat_change_5_steps.columns = ['vesselId', 'avg_lat_change_5_steps']

# Step 4: Calculate the average longitude change for each vesselId based on 1-step and 2-step changes
avg_lon_change_1_steps = train_df.groupby('vesselId')['lon_change_1_steps'].mean().reset_index()
avg_lon_change_1_steps.columns = ['vesselId', 'avg_lon_change_1_steps']

avg_lon_change_2_steps = train_df.groupby('vesselId')['lon_change_2_steps'].mean().reset_index()
avg_lon_change_2_steps.columns = ['vesselId', 'avg_lon_change_2_steps']

avg_lon_change_3_steps = train_df.groupby('vesselId')['lon_change_3_steps'].mean().reset_index()
avg_lon_change_3_steps.columns = ['vesselId', 'avg_lon_change_3_steps']

avg_lon_change_4_steps = train_df.groupby('vesselId')['lon_change_4_steps'].mean().reset_index()
avg_lon_change_4_steps.columns = ['vesselId', 'avg_lon_change_4_steps']

avg_lon_change_5_steps = train_df.groupby('vesselId')['lon_change_5_steps'].mean().reset_index()
avg_lon_change_5_steps.columns = ['vesselId', 'avg_lon_change_5_steps']



# Step 5: Merge the average latitude and longitude changes back into the main dataframe
train_df = pd.merge(train_df, avg_lat_change_1_steps, on='vesselId', how='left')
train_df = pd.merge(train_df, avg_lat_change_2_steps, on='vesselId', how='left')
train_df = pd.merge(train_df, avg_lat_change_3_steps, on='vesselId', how='left')
train_df = pd.merge(train_df, avg_lat_change_4_steps, on='vesselId', how='left')
train_df = pd.merge(train_df, avg_lat_change_5_steps, on='vesselId', how='left')

train_df = pd.merge(train_df, avg_lon_change_1_steps, on='vesselId', how='left')
train_df = pd.merge(train_df, avg_lon_change_2_steps, on='vesselId', how='left')
train_df = pd.merge(train_df, avg_lon_change_3_steps, on='vesselId', how='left')
train_df = pd.merge(train_df, avg_lon_change_4_steps, on='vesselId', how='left')
train_df = pd.merge(train_df, avg_lon_change_5_steps, on='vesselId', how='left')


# Display the final dataframe with the new features
train_df.head()

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,avg_lat_change_1_steps,avg_lat_change_2_steps,avg_lat_change_3_steps,avg_lat_change_4_steps,avg_lat_change_5_steps,avg_lon_change_1_steps,avg_lon_change_2_steps,avg_lon_change_3_steps,avg_lon_change_4_steps,avg_lon_change_5_steps
0,0.031663,0.858217,17.1,-6,316,0,01-08 06:00,7.50361,77.58340,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244
1,0.031707,0.856825,17.3,5,313,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244
2,0.031757,0.854596,16.9,5,312,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244
3,0.031798,0.857660,16.9,6,313,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244
4,0.031838,0.855153,16.3,7,313,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244


In [22]:
train_df = train_df.drop(columns=['lat_change_1_steps', 'lon_change_1_steps',
                                   'lat_change_2_steps', 'lon_change_2_steps',
                                   'lat_change_3_steps', 'lon_change_3_steps',
                                   'lat_change_4_steps', 'lon_change_4_steps',
                                   'lat_change_5_steps', 'lon_change_5_steps',

                                   ])

print(train_df.columns)

train_df.head()

Index(['time', 'cog', 'sog', 'rot', 'heading', 'navstat', 'etaRaw', 'latitude',
       'longitude', 'vesselId', 'portId', 'latitude_1_steps_ago',
       'longitude_1_steps_ago', 'time_position_1_steps_ago',
       'latitude_2_steps_ago', 'longitude_2_steps_ago',
       'time_position_2_steps_ago', 'latitude_3_steps_ago',
       'longitude_3_steps_ago', 'time_position_3_steps_ago',
       'latitude_4_steps_ago', 'longitude_4_steps_ago',
       'time_position_4_steps_ago', 'latitude_5_steps_ago',
       'longitude_5_steps_ago', 'time_position_5_steps_ago',
       'max_lat_change_last_5_steps', 'min_lat_change_last_5_steps',
       'max_long_change_last_5_steps', 'min_long_change_last_5_steps',
       'cog_1_step_ago', 'time_cog_1_step_ago', 'cog_2_steps_ago',
       'time_cog_2_steps_ago', 'hours_passed', 'hour_sin', 'hour_cos',
       'minute_sin', 'minute_cos', 'month_of_the_year', 'week_of_the_year',
       'day_of_the_year', 'day_of_the_month', 'day_of_the_week',
       'hour_of_the_

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,avg_lat_change_1_steps,avg_lat_change_2_steps,avg_lat_change_3_steps,avg_lat_change_4_steps,avg_lat_change_5_steps,avg_lon_change_1_steps,avg_lon_change_2_steps,avg_lon_change_3_steps,avg_lon_change_4_steps,avg_lon_change_5_steps
0,0.031663,0.858217,17.1,-6,316,0,01-08 06:00,7.50361,77.58340,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244
1,0.031707,0.856825,17.3,5,313,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244
2,0.031757,0.854596,16.9,5,312,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244
3,0.031798,0.857660,16.9,6,313,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244
4,0.031838,0.855153,16.3,7,313,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,...,0.073175,0.146541,0.220082,0.293835,0.367811,-0.178895,-0.358466,-0.538683,-0.719606,-0.901244


In [23]:
# Sort by vesselId and time to calculate distances between consecutive positions
test_df = test_df.sort_values(by=['vesselId', 'time']).reset_index(drop=True)

# Step 1: Extract the average latitude and longitude changes from train_df
avg_changes = train_df[['vesselId',  'avg_lat_change_1_steps', 'avg_lon_change_1_steps',
                        'avg_lat_change_2_steps', 'avg_lon_change_2_steps',
                        'avg_lat_change_3_steps', 'avg_lon_change_3_steps',
                        'avg_lat_change_4_steps', 'avg_lon_change_4_steps',
                        'avg_lat_change_5_steps', 'avg_lon_change_5_steps',
                        ]].drop_duplicates()

# Step 2: Merge the average latitude and longitude changes into test_df based on vesselId
test_df = pd.merge(test_df, avg_changes, on='vesselId', how='left')

test_df.head()

,ID,vesselId,time,scaling_factor,port_lat,port_long,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,...,avg_lat_change_1_steps,avg_lon_change_1_steps,avg_lat_change_2_steps,avg_lon_change_2_steps,avg_lat_change_3_steps,avg_lon_change_3_steps,avg_lat_change_4_steps,avg_lon_change_4_steps,avg_lat_change_5_steps,avg_lon_change_5_steps
0,4,61e9f38eb937134a3c4bfd8d,0.349750,0.3,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,-0.000462,-0.001555,-0.000924,-0.003109,-0.001386,-0.004663,-0.001844,-0.006212,-0.002289,-0.007751
1,201,61e9f38eb937134a3c4bfd8d,0.349802,0.3,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,-0.000462,-0.001555,-0.000924,-0.003109,-0.001386,-0.004663,-0.001844,-0.006212,-0.002289,-0.007751
2,583,61e9f38eb937134a3c4bfd8d,0.349904,0.3,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,-0.000462,-0.001555,-0.000924,-0.003109,-0.001386,-0.004663,-0.001844,-0.006212,-0.002289,-0.007751
3,701,61e9f38eb937134a3c4bfd8d,0.349938,0.3,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,-0.000462,-0.001555,-0.000924,-0.003109,-0.001386,-0.004663,-0.001844,-0.006212,-0.002289,-0.007751
4,829,61e9f38eb937134a3c4bfd8d,0.349961,0.3,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,-0.000462,-0.001555,-0.000924,-0.003109,-0.001386,-0.004663,-0.001844,-0.006212,-0.002289,-0.007751


In [24]:
train_df.to_csv('../data/processed_data/train.csv', index=False)
test_df.to_csv("../data/processed_data/test.csv", index=False)